In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
from experiments.jlens_readout_sanity.experiment import (
    Case,
    ExperimentRuntime,
    InterventionSpec,
    ReadoutSpec,
)

CASES = (
    Case(
        key="spider",
        prompt="The number of legs on the animal that spins webs is",
        expected_answers=("8", "eight"),
        readout=ReadoutSpec(
            concepts=("spider",),
            require_capability_gate=True,
        ),
        intervention=InterventionSpec(
            source_surface=" spider",
            target_surface=" ant",
            target_answers=("6", "six"),
        ),
    ),
    Case(
        key="france_capital",
        prompt="The capital of France is the city of",
        expected_answers=("Paris",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Beijing",)),
    ),
    Case(
        key="france_language",
        prompt="Most people in France speak",
        expected_answers=("French",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Chinese",)),
    ),
    Case(
        key="france_continent",
        prompt="France is a country on the continent of",
        expected_answers=("Europe",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Asia",)),
    ),
    Case(
        key="france_currency",
        prompt="The single-word name for the currency now used in France is the",
        expected_answers=("Euro",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Yuan",)),
    ),
)

In [ ]:
import importlib.metadata

import jlens
import torch
import transformers

from experiments.jlens_readout_sanity.constants import (
    LENS_PATH,
    MODEL_PATH,
)
from experiments.jlens_readout_sanity.utils import (
    render_sanity_report,
    run_experiment,
    validate_model_lens,
    write_results,
)
from jlens_reasoning.evaluation import GenerationStatus, ModelOutput

model_path = Path(MODEL_PATH)
lens_path = Path(LENS_PATH)
missing_assets = [path for path in (model_path, lens_path) if not path.exists()]
if missing_assets:
    missing = ", ".join(str(path) for path in missing_assets)
    raise FileNotFoundError(f"Missing Drive assets: {missing}")

causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    local_files_only=True,
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True,
)
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(str(lens_path))
validate_model_lens(model, lens)
model, lens

In [ ]:
@torch.inference_mode()
def forward_next_token(input_ids):
    return causal_lm(input_ids=input_ids, use_cache=False).logits[0, -1]


@torch.inference_mode()
def generate_output(prompt: str) -> ModelOutput:
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(context.device)
    generated = causal_lm.generate(
        input_ids=input_ids,
        do_sample=False,
        max_new_tokens=64,
    )
    generated_ids = generated[0, input_ids.shape[1] :].tolist()
    eos_ids = causal_lm.generation_config.eos_token_id
    eos_token_ids = {eos_ids} if isinstance(eos_ids, int) else set(eos_ids or ())
    generation_status = (
        GenerationStatus.COMPLETE
        if generated_ids and generated_ids[-1] in eos_token_ids
        else GenerationStatus.TRUNCATED
    )
    text_ids = (
        generated_ids[:-1]
        if generation_status is GenerationStatus.COMPLETE
        else generated_ids
    )
    return ModelOutput(
        text=tokenizer.decode(text_ids, skip_special_tokens=True),
        token_ids=tuple(generated_ids),
        token_pieces=tuple(
            tokenizer.decode([token_id], clean_up_tokenization_spaces=False)
            for token_id in generated_ids
        ),
        generation_status=generation_status,
        finish_reason=(
            "eos" if generation_status is GenerationStatus.COMPLETE else "length"
        ),
    )


runtime = ExperimentRuntime(
    model=model,
    lens=lens,
    tokenizer=tokenizer,
    unembedding_weight=causal_lm.get_output_embeddings().weight,
    forward_next_token=forward_next_token,
    generate_output=generate_output,
)
result = run_experiment(cases=CASES, runtime=runtime)

In [ ]:
result.provenance = {
    "project_commit": PROJECT_COMMIT,
    "working_tree_dirty": PROJECT_WORKING_TREE_DIRTY,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "jlens": importlib.metadata.version("jlens"),
}

run_dir = context.runs_dir / "jlens-readout-sanity"
result_path = run_dir / "result.json"
write_results(result_path, result)
print(f"Saved: {result_path}")

In [ ]:
print(render_sanity_report(result))

In [ ]:
if not result.passed:
    raise RuntimeError(
        "Read-and-change sanity checks failed: " + "; ".join(result.failures)
    )

print("All J-Lens read-and-change sanity checks passed.")